# 02 — Pembersihan Data
| | |
|---|---|
| **Nama tim** | Kelompok Alpha |
| **Anggota** | Alfandi Panca Saputra (0110225026) <br>Muhammad Sultan Ridho (0110225099)<br>Dzakwan Abdur Rasyid (0110225036)<br>Mumtaz Abdurrahman (0110225164) |
| **Topik proyek** | Eksplorasi Data Penjualan Bakery |
| **Tanggal** | 9 November 2026 |


In [6]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [1]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

AKAR = Path('.')
RAW = AKAR / 'data' / 'raw'
PROCESSED = AKAR / 'data' / 'processed'
PROCESSED.mkdir(parents=True, exist_ok=True)

In [8]:
BERKAS = '/content/drive/MyDrive/Praktikum EDA/Proyek Tim/Data/raw/Bakery sales (2).csv'

try:
    df_raw = pd.read_csv(BERKAS)
    df = df_raw.copy()
    print(f'baris awal: {len(df_raw)}')
except FileNotFoundError:
    print(f"[ERROR] Berkas tidak ditemukan di jalur: {BERKAS}")
    print("Silakan periksa kembali apakah Google Drive sudah di-mount atau ganti variabel BERKAS dengan jalur yang benar.")

log = []
def catat(langkah, alasan, sebelum, sesudah):
    log.append({'langkah': langkah, 'alasan': alasan,
                'baris_sebelum': sebelum, 'baris_sesudah': sesudah,
                'selisih': sesudah - sebelum})
    print(f'[{len(log):02d}] {langkah}: {sebelum} -> {sesudah} baris ({sesudah-sebelum:+d})')

baris awal: 234005


## Langkah 1 — Drop kolom Unnamed: 0

In [9]:
n0 = len(df)
df = df.drop(columns=['Unnamed: 0'], errors='ignore')
catat('Drop kolom index berlebih', 'Kolom ini duplikasi index pandas', n0, len(df))

[01] Drop kolom index berlebih: 234005 -> 234005 baris (+0)


## Langkah 2 — Konversi Harga (unit_price) menjadi Numerik

In [10]:
n0 = len(df)
df['unit_price'] = df['unit_price'].astype(str).str.replace('€', '').str.replace(',', '.').str.strip()
df['unit_price'] = pd.to_numeric(df['unit_price'])
catat('Konversi unit_price', 'Agar bisa dikalkulasi menjadi total penjualan', n0, len(df))

[02] Konversi unit_price: 234005 -> 234005 baris (+0)


## Langkah 3 — Buang Kuantitas Negatif & Nol (Refund)

In [11]:
n0 = len(df)
df = df[df['Quantity'] > 0]
catat('Buang retur/batal', 'Nilai kuantitas <= 0 adalah retur/kesalahan, tidak relevan untuk analisis tren penjualan normal', n0, len(df))

[03] Buang retur/batal: 234005 -> 232710 baris (-1295)


## Langkah 4 — Hitung Total Penjualan & Gabung Datetime

In [12]:
n0 = len(df)
df['total_sales'] = df['Quantity'] * df['unit_price']
df['datetime'] = pd.to_datetime(df['date'] + ' ' + df['time'])
catat('Tambah fitur turunan', 'Untuk time-series dan visualisasi tren revenue', n0, len(df))

[04] Tambah fitur turunan: 232710 -> 232710 baris (+0)


## Langkah 5 — Samarkan Data (Hashing Ticket)

In [13]:
import hashlib
def samarkan(s):
    return hashlib.sha256(str(s).encode()).hexdigest()[:12]
df['ticket_number'] = df['ticket_number'].map(samarkan)
print(df.columns.tolist())

['date', 'time', 'ticket_number', 'article', 'Quantity', 'unit_price', 'total_sales', 'datetime']


In [14]:
assert (df['Quantity'] > 0).all(), 'masih ada durasi negatif'
assert pd.api.types.is_numeric_dtype(df['unit_price']), 'unit_price bukan numerik'
print(f'LOLOS. {len(df)} baris x {df.shape[1]} kolom siap dianalisis.')

LOLOS. 232710 baris x 8 kolom siap dianalisis.


In [15]:
KELUARAN = PROCESSED / 'data_bersih.csv'
df.to_csv(KELUARAN, index=False)
print('tersimpan:', KELUARAN)

tersimpan: data/processed/data_bersih.csv
